# Input dataset check — clarification (issue #1)

This notebook validates the items required under **issue #1** on the Studio VM.

**Step 1 (this notebook):** list and analyze the input Sentinel product *metadata* and **all data** shipped with the imagery;
answer in particular whether **gain/offset values for radiometric correction are provided with the product**
(issue #1 · C1).

> Run: attach the kernel to JupyterHub/Studio VM. To inspect a specific product, set
> the `S2_INPUT_PRODUCT` environment variable; otherwise the auto-discovery below + `PRODUCT_INDEX`
> is used.

In [ ]:
import os 
import sys
from pathlib import Path

current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")
work_dir = Path(current_dir + "/s2-msi-raw-generator")
os.chdir(work_dir)
print(f"Changed working directory to: {work_dir}")


In [ ]:
# One-time kernel setup — safe to re-run. RESTART kernel after setup.



def _find_repo():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "pyproject.toml").exists() and (base / "s2_msi_raw_generator").is_dir():
            return base
    return None

_need_pkg = _need_deps = False
try:
    import s2_msi_raw_generator  # noqa: F401
except Exception:
    _need_pkg = True
try:
    import zarr        # noqa: F401
    import numpy       # noqa: F401
    import matplotlib  # noqa: F401
except Exception:
    _need_deps = True

_repo = _find_repo()
if _need_pkg and _repo is not None:
    !{sys.executable} -m pip install --quiet -e {_repo} --no-deps
elif _need_pkg:
    print("WARNING: repo root (pyproject.toml + s2_msi_raw_generator) not found; is PYTHONPATH set?")
if _need_deps:
    !{sys.executable} -m pip install --quiet zarr numpy matplotlib
    !{sys.executable} -m pip install --quiet --force-reinstall --no-deps zarr matplotlib
if _need_pkg or _need_deps:
    print("installed/fixed — RESTART kernel and re-run all cells")
else:
    print(f"environment OK (repo: {_repo})")

In [ ]:
# Find / select input product



from s2_msi_raw_generator.io import _open



# --- Configuration ---
# Use S2_INPUT_PRODUCT for full path (.zarr / .zarr.zip); else pick PRODUCT_INDEX from list.
PRODUCT       = os.environ.get("S2_INPUT_PRODUCT", "")
PRODUCT_INDEX = 0

# Root directories to search for input products on Studio VM (existing ones are scanned).
CANDIDATE_ROOTS = [
    os.environ.get("S2_E2ES_L1A", ""),
    os.environ.get("S2_E2ES_L1B", ""),
    "/home/jovyan/data-store/inputs",
    "/home/jovyan/validation-data/data-store/inputs",
    "/home/jovyan/validation-data/e2e-real/inputs",
    str(Path("~/data-store/inputs").expanduser()),
]

def _iter_products(root):
    if root.is_dir() and root.suffix == ".zarr":
        yield root; return
    if root.is_file() and root.name.endswith(".zarr.zip"):
        yield root; return
    if not root.is_dir():
        return
    for p in sorted(root.rglob("*.zarr")):
        if p.is_dir():
            yield p
    for p in sorted(root.rglob("*.zarr.zip")):
        yield p

found, _seen = [], set()
for r in CANDIDATE_ROOTS:
    if not r:
        continue
    for p in _iter_products(Path(r).expanduser()):
        rp = p.resolve()
        if rp in _seen:
            continue
        _seen.add(rp); found.append(p)

print(f"{len(found)} candidate products found:")
for i, p in enumerate(found):
    n = p.name.upper()
    tag = "  <- L1A" if "L1A" in n else "  <- L1B" if "L1B" in n else "  <- L1C" if "L1C" in n else ""
    print(f"  [{i:2d}] {p}{tag}")

if PRODUCT:
    PRODUCT_PATH = Path(PRODUCT).expanduser()
elif found:
    PRODUCT_PATH = found[PRODUCT_INDEX]
else:
    PRODUCT_PATH = None
    print("\n!! No products found. Set S2_INPUT_PRODUCT or edit CANDIDATE_ROOTS.")

print(f"\nSelected product: {PRODUCT_PATH}")

## 1) Product tree — all data shipped with the imagery

Dump every group, array (shape/dtype/approx size), and `attrs` keys at each level.

In [ ]:
# 1) PRODUCT TREE
import numpy as np

assert PRODUCT_PATH is not None, "Select a product first (previous cell)."
root = _open(str(PRODUCT_PATH))
_stats = {"groups": 0, "arrays": 0}

def _fmt_bytes(nb):
    for u in ("B", "KB", "MB", "GB", "TB"):
        if nb < 1024:
            return f"{nb:6.1f}{u}"
        nb /= 1024
    return f"{nb:.1f}PB"

def walk(node, prefix=""):
    a = dict(getattr(node, "attrs", {}))
    if a:
        print(f"{prefix}  . attrs: {list(a.keys())}")
    for name, arr in sorted(getattr(node, "arrays", lambda: [])(), key=lambda kv: kv[0]):
        _stats["arrays"] += 1
        shp, dt = getattr(arr, "shape", "?"), getattr(arr, "dtype", "?")
        try:
            sz = _fmt_bytes(int(np.prod(shp)) * np.dtype(dt).itemsize)
        except Exception:
            sz = "?"
        print(f"{prefix}  [arr] {name:<28} shape={str(shp):<20} dtype={str(dt):<10} {sz}")
    for name, sub in sorted(getattr(node, "groups", lambda: [])(), key=lambda kv: kv[0]):
        _stats["groups"] += 1
        print(f"{prefix}[grp] {name}/")
        walk(sub, prefix + "    ")

print(f"PRODUCT: {PRODUCT_PATH}\n")
print("[grp] / (root)")
walk(root, "    ")
print(f"\nTotal: {_stats['groups']} groups, {_stats['arrays']} arrays")

## 2) Metadata — STAC properties + full attrs tree

Highlighted fields (platform, sensing date, processing baseline, tile/orbit) + full attrs dump.
If `stac_discovery.properties` is empty, `start_datetime`/`platform` are searched in the attrs tree.

In [ ]:
# 2) METADATA
import json

assert PRODUCT_PATH is not None
root = _open(str(PRODUCT_PATH))
attrs = dict(root.attrs)

def find_key(obj, key):
    if isinstance(obj, dict):
        if key in obj and not isinstance(obj[key], (dict, list)):
            return obj[key]
        for v in obj.values():
            r = find_key(v, key)
            if r is not None:
                return r
    elif isinstance(obj, list):
        for v in obj:
            r = find_key(v, key)
            if r is not None:
                return r
    return None

stac  = attrs.get("stac_discovery", {}) or {}
props = stac.get("properties", {}) or {}

print("== Highlighted fields ==")
for label, key in [
    ("platform", "platform"), ("instrument", "instrument"),
    ("sensing start", "start_datetime"), ("sensing end", "end_datetime"),
    ("datetime", "datetime"), ("rel. orbit", "sat:relative_orbit"),
    ("abs. orbit", "sat:absolute_orbit"), ("processing baseline", "processing:version"),
    ("processor", "processing:software"), ("product type", "product:type"),
    ("MGRS tile", "grid:code"),
]:
    val = props.get(key)
    if val is None:
        val = find_key(attrs, key)
    print(f"  {label:22s}: {val}")

print("\n== STAC properties (full) ==")
print(json.dumps(props, indent=2, default=str)[:4000] if props else "  (empty - no stac_discovery.properties)")

print("\n== Full attrs tree (keys) ==")
def walk_attrs(node, prefix="/"):
    a = dict(getattr(node, "attrs", {}))
    if a:
        print(f"  {prefix}")
        for k, v in a.items():
            if isinstance(v, (dict, list)):
                print(f"      {k} [{type(v).__name__}]"[:160])
            else:
                print(f"      {k} = {v}"[:160])
    for name, sub in sorted(getattr(node, "groups", lambda: [])(), key=lambda kv: kv[0]):
        walk_attrs(sub, prefix + name + "/")
walk_attrs(root)

## 3) Is radiometric correction data present? (gain / offset hunt)

At every level (attrs + array names) search for radiometric-correction-related keys:
`gain`, `offset`, `physical_gain`, `quantification`, `radio_add_offset`, `reflectance`,
`solar_irradiance`, `radiometric`, `calibration`, `equalization`, `dark`, `prnu`, `spectral_response` …

This is the direct test for **issue #1 · C1**: if ESA ships gain/offset with the product, the reverse chain
can use them instead of derived `cal_gain`.

In [ ]:
# 3) RADIOMETRIC gain/offset HUNT
assert PRODUCT_PATH is not None
root = _open(str(PRODUCT_PATH))

RADIO_PATTERNS = [
    "gain", "offset", "physical_gain", "quantification",
    "radio_add_offset", "boa_add_offset", "reflectance",
    "solar_irradiance", "u_sun", "sun_earth", "radiance",
    "radiometric", "calibrat", "equaliz", "dark", "prnu",
    "spectral_response", "srf", "abs_cal", "a_coeff", "digital_number",
]

hits = []  # (group_path, key_path, value)

def scan(node, prefix="/"):
    def rec(obj, p):
        if isinstance(obj, dict):
            for k, v in obj.items():
                kp = f"{p}.{k}"
                if any(pat in str(k).lower() for pat in RADIO_PATTERNS):
                    if isinstance(v, dict):
                        val = f"<dict keys={list(v)[:8]}>"
                    elif isinstance(v, list):
                        val = f"<list n={len(v)}> {str(v)[:60]}"
                    else:
                        val = v
                    hits.append((prefix, kp, val))
                rec(v, kp)
        elif isinstance(obj, list):
            for i, v in enumerate(obj[:5]):
                rec(v, f"{p}[{i}]")
    rec(dict(getattr(node, "attrs", {})), "attrs")
    for name, arr in getattr(node, "arrays", lambda: [])():
        if any(pat in name.lower() for pat in RADIO_PATTERNS):
            hits.append((prefix, f"array:{name}", f"shape={getattr(arr,'shape','?')} dtype={getattr(arr,'dtype','?')}"))
    for name, sub in getattr(node, "groups", lambda: [])():
        scan(sub, prefix + name + "/")

scan(root)

print(f"{len(hits)} radiometric-related keys found:\n")
for grp, kp, val in hits:
    print(f"  {grp:<30} {kp:<46} = {str(val)[:70]}")

print("\n" + "=" * 92)
print("VERDICT (issue #1 · C1):")
keys_l = " ".join(kp.lower() for _, kp, _ in hits)
has_gain   = any(k in keys_l for k in ("gain", "physical_gain", "quantification"))
has_offset = any(k in keys_l for k in ("offset", "radio_add_offset", "boa_add_offset"))
print(f"  gain saglanmis?   {'EVET' if has_gain else 'HAYIR'}")
print(f"  offset saglanmis? {'EVET' if has_offset else 'HAYIR'}")
print("  -> If values come with the product, use ESA gain/offset instead of derived cal_gain in reverse chain")
print("     usable (issue #1 C1). Confirm values and units above.")

## Summary & next step

- **Product tree** (section 1) = all data layers shipped with the imagery.
- **Metadata** (section 2) = sensing date + processing baseline → required for issue #1 · D (temporal validity);
  this is the reference date to compare against calibration file validity windows.
- **Radiometric hunt** (section 3) = do gain/offset come with the product, or is external GIPP/derivation needed
  (issue #1 · C1).

**Next step (separate cell/notebook):** compare the found sensing date against GIPP/PSF file
production + validity windows and emit a match flag (issue #1 · D4).

## 4) Scan all products — WHERE is metadata/gain/offset? (group **+ array** attrs)

`PDI_MSI_S2_L1A.zarr` (index 0, the pipeline's actual input) **carries no metadata** — so
sections 1–3 returned empty. This cell scans all 13 products and searches **both group and array attrs**
for radiometric keys; finds where gain/offset actually live (if anywhere).

In [ ]:
# 4) Scan ALL products (group + array attrs)
import os
from pathlib import Path

from s2_msi_raw_generator.io import _open

CANDIDATE_ROOTS = [
    "/home/jovyan/data-store/inputs",
    "/home/jovyan/validation-data/data-store/inputs",
    str(Path("~/data-store/inputs").expanduser()),
]
RADIO_PATTERNS = [
    "gain", "offset", "physical_gain", "quantification", "radio_add_offset",
    "boa_add_offset", "reflectance", "solar_irradiance", "radiance", "radiometric",
    "calibrat", "equaliz", "dark", "prnu", "spectral_response", "srf", "abs_cal",
    "a_coeff", "alpha", "beta", "noise", "scale_factor", "add_offset",
]

def _products():
    out, seen = [], set()
    for r in CANDIDATE_ROOTS:
        root = Path(r)
        if not root.is_dir():
            continue
        for p in list(sorted(root.rglob("*.zarr"))) + list(sorted(root.rglob("*.zarr.zip"))):
            if (p.is_dir() or p.name.endswith(".zip")) and p.resolve() not in seen:
                seen.add(p.resolve()); out.append(p)
    return out

def _match(s):
    s = str(s).lower()
    return any(pat in s for pat in RADIO_PATTERNS)

def _analyze(node, acc, prefix="/"):
    ga = dict(getattr(node, "attrs", {}))
    acc["gattrs"] += len(ga)
    def rec(obj, p):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if _match(k):
                    val = v if not isinstance(v, (dict, list)) else f"<{type(v).__name__}>"
                    acc["hits"].append((prefix, f"gattr:{p}.{k}", val))
                rec(v, f"{p}.{k}")
        elif isinstance(obj, list):
            for i, v in enumerate(obj[:5]):
                rec(v, f"{p}[{i}]")
    rec(ga, "g")
    for name, arr in getattr(node, "arrays", lambda: [])():
        acc["arrs"] += 1
        aa = dict(getattr(arr, "attrs", {}))
        acc["aattrs"] += len(aa)
        if _match(name):
            acc["hits"].append((prefix, f"array:{name}", f"shape={getattr(arr, 'shape', '?')}"))
        for k, v in aa.items():
            if _match(k):
                val = v if not isinstance(v, (dict, list)) else f"<{type(v).__name__}>"
                acc["hits"].append((prefix, f"arrattr:{name}.{k}", val))
    for name, sub in getattr(node, "groups", lambda: [])():
        acc["grps"] += 1
        _analyze(sub, acc, prefix + name + "/")

prods = _products()
print(f"{len(prods)} products scanning (group + array attrs)\n")
print(f"{'#':>2}  {'grp':>4} {'arr':>4} {'gAttr':>6} {'aAttr':>6} {'radio':>6}  product")
print("-" * 108)
summary = []
for i, p in enumerate(prods):
    acc = {"grps": 0, "arrs": 0, "gattrs": 0, "aattrs": 0, "hits": []}
    try:
        _analyze(_open(str(p)), acc)
    except Exception as e:
        print(f"[{i:2d}] ERROR: {str(e)[:60]}  {p.name}")
        continue
    summary.append((i, p, acc))
    rel = (str(p).replace("/home/jovyan/data-store/inputs/", "")
                 .replace("/home/jovyan/validation-data/data-store/inputs/", "VAL/"))
    print(f"[{i:2d}] {acc['grps']:>4} {acc['arrs']:>4} {acc['gattrs']:>6} {acc['aattrs']:>6} {len(acc['hits']):>6}  {rel}")

print("\n" + "=" * 108)
print("Products CONTAINING radiometric keys (detail):")
any_hit = False
for i, p, acc in summary:
    if acc["hits"]:
        any_hit = True
        print(f"\n[{i}] {p.name}  ({len(acc['hits'])} isabet)")
        for grp, kp, val in acc["hits"][:40]:
            print(f"     {grp:<26} {kp:<38} = {str(val)[:56]}")
if not any_hit:
    print("  No product has radiometric gain/offset/calibration keys.")
    print("  -> ESA input products do not carry radiometric coefficients; values come from separate GIPP/ADF (issue #1 C1).")

## 5) NUC gain/offset — ESA source (GIPP R2EQOG)

The input product does not carry NUC gain/offset **values** (only `*_proc` flags). Real ESA NUC
data lives in **GIPP R2EQOG**: per-pixel **dark = NUC offset**, **rel-response (PRNU) = NUC gain**
(VNIR cubic `C` / SWIR bilinear `A1`). This cell reads GIPP with the repo parser and extracts values —
is ESA NUC data present, what is its range? (issue #1 · C1/D).

> Note: the reverse chain uses these via `adf.from_gipp`; but delivered `caldb/nuc.zarr` is
> **derived** from synthetic dark+diffuser (`caldb.derive_from_acquisitions`), not from ESA.

In [ ]:
# 5) ESA NUC source: GIPP R2EQOG (dark=offset, PRNU=gain)
import os
from pathlib import Path

import numpy as np

from s2_msi_raw_generator import gipp, sensor

GIPP_CANDS = [
    os.environ.get("S2_E2ES_GIPP_DIR", ""),
    "/home/jovyan/data-store/inputs/s2-sensor/GIPP",
    "/home/jovyan/validation-data/data-store/inputs/s2-sensor/GIPP",
    str(Path("~/data-store/inputs/s2-sensor/GIPP").expanduser()),
]
gipp_dir = next((d for d in GIPP_CANDS if d and Path(d).is_dir()), None)
print("GIPP directory:", gipp_dir)

if gipp_dir is None:
    print("!! GIPP directory not found — set S2_E2ES_GIPP_DIR.")
else:
    import re

    files = sorted(p for p in Path(gipp_dir).rglob("*") if p.is_file())
    # S2 GIPP adi:  ..._GIP_<TYPE>_MPC__<CREATION>_V<VALID_START>_<VALID_STOP>_...
    NAME_RE = re.compile(
        r"GIP_(?P<type>\w+?)_MPC__(?P<creation>\d{8}T\d{6})"
        r"_V(?P<vstart>\d{8}T\d{6})_(?P<vstop>\d{8})"
    )

    def _d(ts):
        return f"{ts[0:4]}-{ts[4:6]}-{ts[6:8]}"

    print(f"\n{len(files)} GIPP files — PRODUCTION (creation) + validity (parsed from name):")
    print(f"  {'type':8} {'PRODUCTION':12} {'VALID-FROM':12} {'VALID-TO':12}")
    print("  " + "-" * 50)
    for f in files:
        m = NAME_RE.search(f.name)
        if m:
            print(f"  {m['type']:8} {_d(m['creation']):12} {_d(m['vstart']):12} {_d(m['vstop']):12}")
        else:
            print(f"  (name not parsed) {f.name}")

    print("\nR2EQOG -> NUC (detector 1):  offset=dark[LSB], gain=PRNU rel-response")
    print(f"  {'band':5} {'model':9} {'npix':>6}   {'dark(offset) min/mean/max':<30}   {'gain(PRNU) min/mean/max':<30}")
    print("  " + "-" * 92)
    ok = 0
    for b in sensor.BANDS:
        try:
            be = gipp.read_r2eqog_band(gipp_dir, b)
            de = be.detectors[1]
            d, g = np.asarray(de.dark, float), np.asarray(de.rel_gain, float)
            print(f"  {b:5} {de.model:9} {d.size:>6}   "
                  f"{d.min():8.2f}/{d.mean():8.2f}/{d.max():8.2f}      "
                  f"{g.min():8.4f}/{g.mean():8.4f}/{g.max():8.4f}")
            ok += 1
        except Exception as e:
            print(f"  {b:5}  ERROR: {str(e)[:70]}")
    print(f"\n{ok}/13 bands: R2EQOG NUC gain+offset read.")
    print("VERDICT: ESA NUC data in GIPP R2EQOG is " + ("PRESENT." if ok else "UNREADABLE."))
    print("  -> reverse chain uses these; delivered nuc.zarr is synthetically derived (issue #1).")


## 6) `validation-data/ci_test_data/s02` review

Is there useful data in this folder? Inventory (top level + extension distribution) + summary per zarr product
(sensing date, radiometric hits) + GIPP dates + README/sidecar preview. Override path with
via `S2_CI_ROOT`.

In [ ]:
# 6) ci_test_data/s02 — inventory + useful-data scan
import os
import re
from collections import Counter
from pathlib import Path

from s2_msi_raw_generator.io import _open

ROOT = Path(os.environ.get("S2_CI_ROOT", "/home/jovyan/validation-data/ci_test_data/s02"))
print("ROOT:", ROOT, "" if ROOT.exists() else "  <-- MISSING / inaccessible")

RADIO = ["gain", "offset", "physical_gain", "quantification", "radio_add_offset", "reflectance",
         "solar_irradiance", "radiance", "radiometric", "calibrat", "equaliz", "dark", "prnu",
         "spectral_response", "srf", "alpha", "beta", "noise", "nuc"]

def _match(s):
    s = str(s).lower()
    return any(p in s for p in RADIO)

def _find_key(obj, key):
    if isinstance(obj, dict):
        if key in obj and not isinstance(obj[key], (dict, list)):
            return obj[key]
        for v in obj.values():
            r = _find_key(v, key)
            if r is not None:
                return r
    elif isinstance(obj, list):
        for v in obj:
            r = _find_key(v, key)
            if r is not None:
                return r
    return None

def _summ(node, acc):
    ga = dict(getattr(node, "attrs", {}))
    acc["ga"] += len(ga)
    for k in ga:
        if _match(k):
            acc["radio"] += 1
    for name, arr in getattr(node, "arrays", lambda: [])():
        acc["arr"] += 1
        aa = dict(getattr(arr, "attrs", {}))
        acc["aa"] += len(aa)
        if _match(name) or any(_match(k) for k in aa):
            acc["radio"] += 1
    for name, sub in getattr(node, "groups", lambda: [])():
        acc["grp"] += 1
        _summ(sub, acc)

def _du(p):
    try:
        return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
    except Exception:
        return 0

if not ROOT.exists():
    print("!! Path missing. Provide another path via S2_CI_ROOT.")
else:
    print("\n== top-level content ==")
    for e in sorted(ROOT.iterdir()):
        kind = "dir " if e.is_dir() else "file"
        sz = _du(e) if e.is_dir() else e.stat().st_size
        print(f"  {kind}  {sz / 1e6:9.1f} MB  {e.name}")

    exts = Counter(p.suffix.lower() for p in ROOT.rglob("*") if p.is_file())
    print("\n== extension distribution ==")
    for ext, n in exts.most_common(20):
        print(f"  {ext or '(no ext)':12} {n}")

    prods = sorted(set(list(ROOT.rglob("*.zarr")) + list(ROOT.rglob("*.zarr.zip"))))
    prods = [p for p in prods if p.is_dir() or p.name.endswith(".zip")]
    print(f"\n== {len(prods)} zarr products (summary) ==")
    for p in prods:
        try:
            g = _open(str(p))
            acc = {"grp": 0, "arr": 0, "ga": 0, "aa": 0, "radio": 0}
            _summ(g, acc)
            attrs = dict(g.attrs)
            dt = _find_key(attrs, "start_datetime") or _find_key(attrs, "datetime")
            plat = _find_key(attrs, "platform")
            print(f"  {p.relative_to(ROOT)}")
            print(f"      grp={acc['grp']} arr={acc['arr']} gAttr={acc['ga']} aAttr={acc['aa']} "
                  f"radioHit={acc['radio']}  sensing={dt}  platform={plat}")
        except Exception as e:
            print(f"  {p.relative_to(ROOT)}  ERROR: {str(e)[:60]}")

    NAME_RE = re.compile(r"GIP_(?P<type>\w+?)_MPC__(?P<c>\d{8})T\d{6}_V(?P<vs>\d{8})T\d{6}_(?P<ve>\d{8})")
    gipp = [p for p in ROOT.rglob("*") if p.is_file() and "GIP_" in p.name]
    if gipp:
        print(f"\n== {len(gipp)} GIPP files (unique type+production) ==")
        seen = set()
        for f in sorted(gipp):
            m = NAME_RE.search(f.name)
            if m and (m["type"], m["c"]) not in seen:
                seen.add((m["type"], m["c"]))
                fmt = lambda t: f"{t[:4]}-{t[4:6]}-{t[6:8]}"
                print(f"  {m['type']:8} production={fmt(m['c'])} valid={fmt(m['vs'])} -> {fmt(m['ve'])}")

    txts = [p for p in ROOT.rglob("*")
            if p.is_file() and p.suffix.lower() in (".md", ".txt", ".json", ".xml")
            and p.stat().st_size < 40000]
    if txts:
        print(f"\n== {len(txts)} text/sidecar (first 4 previews) ==")
        for t in txts[:4]:
            print(f"\n  --- {t.relative_to(ROOT)} ---")
            body = t.read_text(errors="replace")[:700]
            print("  " + body.replace("\n", "\n  "))

## 7) Real SAFE acquisitions: sensing date ↔ applied GIPP (temporal-validity ground truth)

`ci_test_data/s02` contains real S2 SAFE products. Each product's **datastrip metadata** carries the
**applied GIPP list** (names encode production/validity dates). This cell extracts per SAFE
sensing date + applied R2EQOG (and other GIPPs) → shows which calibration ESA used on real
acquisitions and whether it matches the input date (issue #1 · D).

In [ ]:
# 7) SAFE: sensing ↔ applied GIPP
import os
import re
from pathlib import Path

ROOT = Path(os.environ.get("S2_CI_ROOT", "/home/jovyan/validation-data/ci_test_data/s02"))

def _d(t):
    return f"{t[:4]}-{t[4:6]}-{t[6:8]}" if t else "?"

def _sensing(name):
    v = re.search(r"_V(\d{8})T", name)          # OPER PRD: _V<sensing>T
    if v:
        return v.group(1)
    m = re.search(r"MSIL[0-9][A-C]?_(\d{8})T", name)  # kompakt: MSIL1B_<sensing>T
    return m.group(1) if m else None

GIP_RE = re.compile(
    r"S2[AB]_OPER_GIP_(?P<type>\w+?)_\w+?__(?P<c>\d{8})T\d{6}"
    r"(?:_V(?P<vs>\d{8})T\d{6}_(?P<ve>\d{8}))?"
)

if not ROOT.exists():
    print("!! Path missing:", ROOT)
else:
    safes = sorted(p for p in ROOT.iterdir() if p.is_dir() and p.name.endswith(".SAFE"))
    print(f"{len(safes)} SAFE products\n")
    for s in safes:
        lvl = next((L for L in ("L1A", "L1B", "L1C", "L2A", "L0") if L in s.name), "?")
        sens = _sensing(s.name)
        xmls = list(s.rglob("MTD_DS.xml")) or list(s.rglob("*DS*.xml"))[:2]
        gipps = {}
        for x in xmls:
            try:
                txt = x.read_text(errors="replace")
            except Exception:
                continue
            for m in GIP_RE.finditer(txt):
                gipps.setdefault(m["type"], (m["c"], m["vs"], m["ve"]))
        print(f"[{lvl}] sensing={_d(sens)}   {s.name[:70]}")
        r2 = gipps.get("R2EQOG")
        if r2:
            gap = ""
            if sens and r2[0]:
                gap = f"   (delta: sensing {sens[:4]} vs GIPP production {r2[0][:4]})"
            print(f"      R2EQOG applied: production={_d(r2[0])} valid={_d(r2[1])}->{_d(r2[2])}{gap}")
        if gipps:
            print("      all GIPP: " + ", ".join(f"{t}({_d(c)})" for t, (c, vs, ve) in sorted(gipps.items())))
        else:
            print("      (no GIPP reference in datastrip)")
        print()

## 8) Real or synthetic? — provenance clarification

Are input product pixels (`PDI_MSI_S2_L1A`, `20240403` L1A/L1B) a real Earth scene,
or synthetic/test data? Two definitive tests:
1. **Imagery** — real scene (coast, fields, clouds, structures) vs flat/gradient/noise.
2. **Lineage metadata** — datatake / datastrip / downlink / facility / footprint / source_product izleri.

Additionally: `20240403T000000` is midnight UTC → S2 does not acquire optical imagery, so the **name is certainly
a placeholder**. Decision is in the pixels.

In [ ]:
# 8) Provenance: imagery + statistics + lineage metadata
import os
from pathlib import Path

import numpy as np

from s2_msi_raw_generator.io import _open

targets = {
    "PDI (pipeline input)":
        "/home/jovyan/data-store/inputs/PDI_MSI_S2_L1A.zarr",
    "20240403 L1A":
        "/home/jovyan/data-store/inputs/public-data/level-1/S02MSIL1A_20240403T000000_0001_A123_T000.zarr",
    "20240403 L1B":
        "/home/jovyan/data-store/inputs/public-data/level-1/S02MSIL1B_20240403T000000_0001_A123_T000.zarr",
}

try:
    import matplotlib.pyplot as plt
    have_plt = True
except Exception:
    have_plt = False
    print("matplotlib missing — statistics only. (pip install matplotlib)")

def first_image(g):
    def walk(node, pfx=""):
        for name, arr in getattr(node, "arrays", lambda: [])():
            if getattr(arr, "ndim", 0) == 2 and (
                name.endswith("image") or name.startswith("band") or name == "img"
            ):
                return pfx + name, arr
        for name, sub in getattr(node, "groups", lambda: [])():
            r = walk(sub, pfx + name + "/")
            if r:
                return r
        return None
    return walk(g)

# --- A) IMAGERY + statistics (real scene?) ---
for label, path in targets.items():
    print("=" * 84)
    print(label, "\n ", path)
    if not Path(path).exists():
        print("  (missing)")
        continue
    g = _open(path)
    hit = first_image(g)
    if not hit:
        print("  (2D image not found)")
        continue
    name, arr = hit
    win = np.asarray(arr[: min(1024, arr.shape[0])])
    print(f"  array: {name}  shape={arr.shape}  dtype={arr.dtype}")
    print(f"  statistics: min={win.min():.1f} max={win.max():.1f} "
          f"mean={win.mean():.1f} std={win.std():.1f}  (very low std => flat => synthetic hint)")
    if have_plt:
        lo, hi = np.percentile(win, (2, 98))
        plt.figure(figsize=(8, 5))
        plt.imshow(np.clip(win, lo, max(hi, lo + 1e-9)), cmap="gray", aspect="auto",
                   interpolation="nearest")
        plt.title(f"{label}: {name}  [{lo:.0f}-{hi:.0f}]  {win.shape}")
        plt.xlabel("column"); plt.ylabel("row"); plt.colorbar(label="DN")
        plt.tight_layout(); plt.show()

# --- B) LINEAGE: provenance traces in 20240403 L1B other_metadata ---
print("\n" + "=" * 84)
print("LINEAGE METADATA (20240403 L1B) — datatake/datastrip/downlink/facility/footprint/source:")
p = targets["20240403 L1B"]
if Path(p).exists():
    attrs = dict(_open(p).attrs)
    LIN = ["source", "lineage", "datatake", "datastrip", "downlink", "facility", "granule",
           "product_uri", "product_id", "footprint", "coordinate", "sensing", "orbit",
           "platform", "processor", "baseline", "processing_time", "generation", "reception"]
    hits = 0
    def walk_meta(obj, pfx=""):
        global hits
        if isinstance(obj, dict):
            for k, v in obj.items():
                kp = f"{pfx}.{k}"
                if any(h in str(k).lower() for h in LIN) and not isinstance(v, (dict, list)):
                    print(f"  {kp} = {v}")
                    hits += 1
                walk_meta(v, kp)
        elif isinstance(obj, list):
            for i, v in enumerate(obj[:3]):
                walk_meta(v, f"{pfx}[{i}]")
    walk_meta(attrs)
    if hits == 0:
        print("  (no lineage/source keys — no provenance trail)")

# --- C) provenance document in data-store (README/PROVENANCE/manifest) ---
print("\n" + "=" * 84)
print("PROVENANCE DOCUMENT search (under inputs: README/PROVENANCE/manifest/*.json):")
inp = Path("/home/jovyan/data-store/inputs")
found_doc = False
if inp.is_dir():
    for pat in ("README*", "PROVENANCE*", "*manifest*", "*.md"):
        for f in sorted(inp.rglob(pat)):
            if f.is_file() and f.stat().st_size < 60000:
                found_doc = True
                print(f"\n  --- {f.relative_to(inp)} ---")
                print("  " + f.read_text(errors="replace")[:900].replace("\n", "\n  "))
if not found_doc:
    print("  (no provenance document found)")

## 9) Are L0→L1 processing parameters (GIPP/calibration) recorded in the product?

Does product metadata record **which calibration parameters / GIPPs were applied** to produce it?
In SAFE this lives in datastrip `GIPP_List`. In EOPF
zarr? This cell searches processing-provenance keys + **`GIP_` filename** references in metadata
→ definitive answer on whether an L0→L1 calibration trail exists.

In [ ]:
# 9) Processing provenance: GIPP/aux/calibration reference present?
import json
import re
from pathlib import Path

from s2_msi_raw_generator.io import _open

paths = {
    "20240403 L1B": "/home/jovyan/data-store/inputs/public-data/level-1/S02MSIL1B_20240403T000000_0001_A123_T000.zarr",
    "20240403 L1A": "/home/jovyan/data-store/inputs/public-data/level-1/S02MSIL1A_20240403T000000_0001_A123_T000.zarr",
    "PDI L1A":      "/home/jovyan/data-store/inputs/PDI_MSI_S2_L1A.zarr",
    "L0P 20230216": "/home/jovyan/data-store/inputs/public-data/level-0/S02MSIL0P_20240403T000000_0001_A123_T000.zarr",
}

PROV = ["gip", "gipp", "auxiliary", "aux_data", "adf", "processing", "baseline", "processor",
        "calibrat", "r2eqog", "r2depi", "r2para", "blindp", "r2crco", "r2binn",
        "reference", "ancillary", "lut", "table", "software", "creation"]

def walk(obj, pfx, hits):
    if isinstance(obj, dict):
        for k, v in obj.items():
            kp = f"{pfx}.{k}"
            if any(p in str(k).lower() for p in PROV):
                if isinstance(v, dict):
                    hits.append((kp, f"<dict keys={list(v)[:10]}>"))
                elif isinstance(v, list):
                    hits.append((kp, f"<list n={len(v)}>"))
                else:
                    hits.append((kp, v))
            walk(v, kp, hits)
    elif isinstance(obj, list):
        for i, v in enumerate(obj[:5]):
            walk(v, f"{pfx}[{i}]", hits)

for label, path in paths.items():
    print("=" * 88)
    print(label, "\n ", path)
    if not Path(path).exists():
        print("  (missing)")
        continue
    attrs = dict(_open(path).attrs)

    # A) processing/calibration keys
    hits = []
    walk(attrs, "attrs", hits)
    if hits:
        print(f"  {len(hits)} processing/calib. keys:")
        for kp, v in hits[:30]:
            print(f"     {kp} = {str(v)[:80]}")
    else:
        print("  no processing/calib. keys")

    # B) GIP_ filename reference inside metadata (definitive test)
    blob = json.dumps(attrs, default=str)
    gips = sorted(set(re.findall(r"S2[AB]_OPER_GIP_\w+", blob)))
    print(f"\n  >>> GIPP filename reference: {gips if gips else 'NONE — applied GIPP list not recorded'}")
    # possible ADF/aux filename reference
    adfs = sorted(set(re.findall(r"S2[AB]_OPER_(?:AUX|GIP|MSI)_\w+", blob)))[:10]
    if adfs:
        print(f"  >>> other OPER file references: {adfs}")
    print()

## 10) Other calibration data hunt — validation-data + shared (all subfolders)

Recursively scans `/home/jovyan/validation-data` and `/home/jovyan/shared`. Goal: besides the pipeline's
bundled R2EQOG (2020-03-10), are there **other epoch GIPPs** (especially R2EQOG/R2ABCA) and
other calibration artifacts (nuc/caldb zarr, PSF, SRF, dark, diffuser)? Lists findings **by type
+ production epoch**; flags R2EQOG matching our input acquisition date (**2023-02-16**).

In [ ]:
# 10) validation-data + shared: standalone GIPP / calibration file hunt (recursive)
import os
import re
from collections import defaultdict
from pathlib import Path

ROOTS = [Path("/home/jovyan/validation-data"), Path("/home/jovyan/shared")]

def _d(t):
    return f"{t[:4]}-{t[4:6]}-{t[6:8]}" if t else "?"

# GIPP filename: S2A_OPER_GIP_<TYPE6>_<SITE>__<creation>T..[_V<vstart>T.._<vstop>]
GIP_RE = re.compile(
    r"S2[AB]_OPER_GIP_(?P<type>[A-Z0-9]{6})_[A-Za-z0-9]+_{1,2}(?P<c>\d{8})T\d{6}"
    r"(?:_V(?P<vs>\d{8})T\d{6}_(?P<ve>\d{8}))?"
)
RADIO = {"R2EQOG", "R2ABCA"}  # radiometric pair we care about
CAL_HINT = re.compile(r"(nuc|caldb|diffus|darksig|r2eqog|r2abca|physical_gain|\bsrf\b|psf)", re.I)
TARGET_LO, TARGET_HI = "20220801", "20230801"  # reasonable window for 2023-02 acquisition

# type -> {(c, vs, ve): sample_path}
epochs = defaultdict(dict)
cal_files = []          # non-GIPP calibration artifacts
loose_radio = []        # mentions R2EQOG/R2ABCA but regex miss
n_files = 0
n_dirs = 0

for root in ROOTS:
    print("=" * 90)
    if not root.exists():
        print("!! missing:", root); continue
    print("scanning:", root)
    for dp, dns, fns in os.walk(root, followlinks=False):
        n_dirs += 1
        # try both file and folder names (GIPP sometimes .SAFE-like folder)
        for name in list(fns) + list(dns):
            n_files += 1
            m = GIP_RE.search(name)
            if m:
                key = (m["c"], m["vs"], m["ve"])
                epochs[m["type"]].setdefault(key, str(Path(dp) / name))
            elif ("R2EQOG" in name or "R2ABCA" in name):
                loose_radio.append(str(Path(dp) / name))
            elif CAL_HINT.search(name) and name != root.name:
                cal_files.append(str(Path(dp) / name))

print("\n" + "=" * 90)
print(f"scan complete: {n_dirs} folders, {n_files} names inspected\n")

if not epochs:
    print(">>> No GIPP files found (OPER_GIP_ pattern).")
else:
    print(f">>> {len(epochs)} distinct GIPP types found.\n")
    # radiometric pair first — ALL epochs
    for t in sorted(RADIO):
        if t not in epochs:
            print(f"[{t}]  not found\n"); continue
        eps = sorted(epochs[t], key=lambda k: k[0] or "")
        print(f"[{t}]  {len(eps)} epochs:")
        for c, vs, ve in eps:
            tgt = "   <<< MATCHES 2023-02 acquisition" if (c and TARGET_LO <= c <= TARGET_HI) else ""
            print(f"     production={_d(c)}  valid={_d(vs)}->{_d(ve)}{tgt}")
        ex = epochs[t][eps[0]]
        print(f"     sample path: {ex}\n")
    # other GIPP types — epoch count summary
    others = sorted(set(epochs) - RADIO)
    if others:
        print("other GIPP types (type: epoch count, production dates):")
        for t in others:
            eps = sorted(epochs[t], key=lambda k: k[0] or "")
            ds = ", ".join(_d(c) for c, _, _ in eps)
            print(f"  {t}: {len(eps)} epochs -> {ds}")

print("\n" + "=" * 90)
print("Non-GIPP calibration artifacts (nuc/caldb/diffuser/dark/psf/srf/physical_gain):")
if cal_files:
    for p in sorted(set(cal_files))[:40]:
        print("  ", p)
    if len(set(cal_files)) > 40:
        print(f"   ... (+{len(set(cal_files))-40} daha)")
else:
    print("   (none)")

if loose_radio:
    print("\nFiles mentioning R2EQOG/R2ABCA but not matching pattern:")
    for p in sorted(set(loose_radio))[:30]:
        print("  ", p)

print("\n" + "=" * 90)
print("RESULT: pipeline bundle=R2EQOG 2020-03-10. If an R2EQOG epoch appears above in 2022-08..2023-08")
print("then ESA-compatible data for our 2023-02-16 acquisition is AVAILABLE.")

## 11) PDI_MSI_S2_L1A acquisition date — embedded date + pixel matching

PDI product has no date in name or metadata. Two tests: (A) recursively scan all attrs (group+array)
for **any ISO date** value; (B) match PDI B01 pixels against dated real products
(2023-02-16 L0 / 20240403 L1A B01) — **same scene?** If matched, we infer acquisition date indirectly.

In [ ]:
# 11) PDI acquisition date: embedded-date scan + pixel matching
import re
from pathlib import Path

import numpy as np

from s2_msi_raw_generator.io import _open

PDI = "/home/jovyan/data-store/inputs/PDI_MSI_S2_L1A.zarr"
DATE_RE = re.compile(r"20\d{2}[-/]?[01]\d[-/]?[0-3]\d(?:[T ][0-2]\d[:.]?[0-5]\d)?")

def all_attrs(g, pfx=""):
    out = {}
    try:
        out.update({f"{pfx}{k}": v for k, v in dict(g.attrs).items()})
    except Exception:
        pass
    for nm, sub in getattr(g, "groups", lambda: [])():
        out.update(all_attrs(sub, f"{pfx}{nm}/"))
    for nm, arr in getattr(g, "arrays", lambda: [])():
        try:
            for k, v in dict(arr.attrs).items():
                out[f"{pfx}{nm}@{k}"] = v
        except Exception:
            pass
    return out

print("=" * 84, "\nA) Date values in PDI attrs:")
g = _open(PDI)
attrs = all_attrs(g)
print(f"  total {len(attrs)} attr keys")
date_hits = []
for k, v in attrs.items():
    for m in DATE_RE.findall(str(v)):
        date_hits.append((k, str(v)[:60], m))
if date_hits:
    for k, v, m in date_hits[:30]:
        print(f"  {k} = {v}   -> date: {m}")
else:
    print("  >>> NO date in any attr — PDI carries no embedded acquisition date.")
# show all keys even without dates (if few)
if len(attrs) <= 25:
    print("  all attrs:")
    for k, v in attrs.items():
        print(f"     {k} = {str(v)[:70]}")

print("\n" + "=" * 84, "\nB) Which dated product do PDI B01 pixels match?")

def first_b01(path):
    node = _open(path)
    def walk(g, pfx=""):
        for nm, arr in getattr(g, "arrays", lambda: [])():
            if getattr(arr, "ndim", 0) == 2 and ("B01" in pfx or "B01" in nm or "b01" in (pfx+nm).lower()):
                return pfx + nm, arr
        for nm, sub in getattr(g, "groups", lambda: [])():
            r = walk(sub, pfx + nm + "/")
            if r:
                return r
        return None
    return walk(node)

pdi_hit = first_b01(PDI)
if not pdi_hit:
    print("  PDI B01 not found");
else:
    pn, pa = pdi_hit
    P = np.asarray(pa[:512, :512], dtype=float)
    print(f"  PDI B01: {pn} shape={pa.shape} mean={P.mean():.1f} std={P.std():.1f}")
    cands = {
        "20240403 L1A": "/home/jovyan/data-store/inputs/public-data/level-1/S02MSIL1A_20240403T000000_0001_A123_T000.zarr",
        "20230216 L0":  "/home/jovyan/data-store/inputs/public-data/level-0/S02MSIL0__20230216T182840_0001_A123_T000.zarr",
        "20240403 L0P": "/home/jovyan/data-store/inputs/public-data/level-0/S02MSIL0P_20240403T000000_0001_A123_T000.zarr",
    }
    for label, path in cands.items():
        if not Path(path).exists():
            print(f"  [{label}] missing"); continue
        h = first_b01(path)
        if not h:
            print(f"  [{label}] no B01"); continue
        cn, ca = h
        C = np.asarray(ca[:512, :512], dtype=float)
        same_shape = ca.shape == pa.shape
        # scale may differ (raw vs calibrated) → normalized correlation
        if C.shape == P.shape:
            a = (P - P.mean()) / (P.std() + 1e-9)
            b = (C - C.mean()) / (C.std() + 1e-9)
            corr = float((a * b).mean())
            ident = bool(np.allclose(P, C))
            tag = "SAME SCENE" if corr > 0.95 else ("similar" if corr > 0.6 else "different")
        else:
            corr, ident, tag = float("nan"), False, "shape mismatch"
        print(f"  [{label}] {cn} shape={ca.shape} mean={C.mean():.1f} "
              f"corr={corr:.3f} exact={ident} -> {tag}")
    print("\n  Note: sensing date of product marked 'SAME SCENE' = PDI's real acquisition date.")

## 12) VM S3 access probe — can the VM reach dpr-l0-input with its own identity?

Goal: before downloading from OVH bucket (`dpr-l0-input`) to `/home/jovyan/validation-data/data-store`,
verify the VM's **own** access. This cell: (A) free space on target disk, (B) available tools
(boto3/s3fs/rclone/aws), (C) S3 credential sources on VM (env / ~/.aws / rclone.conf —
presence only, no values printed), (D) **live listing attempt** on bucket (VM default credential chain).
Local secrets.json is NOT used — only the VM's own access is tested.

In [ ]:
# 12) VM S3 access probe (no local secret)
import os
import shutil
from pathlib import Path

ENDPOINT = "https://s3.sbg.io.cloud.ovh.net"
REGION = "sbg"
BUCKET = "dpr-l0-input"
PREFIX = "L0/Validation/S2/"
TARGET = Path("/home/jovyan/validation-data/data-store")

print("A) TARGET DISK")
probe = TARGET if TARGET.exists() else TARGET.parent if TARGET.parent.exists() else Path("/home/jovyan")
try:
    tot, used, free = shutil.disk_usage(probe)
    print(f"  {probe}: free={free/2**30:.1f} GiB / total={tot/2**30:.1f} GiB")
    print(f"  167 GiB download: {'OK' if free/2**30 > 180 else 'RISKY/INSUFFICIENT (>180 GiB recommended)'}")
except Exception as e:
    print("  disk unreadable:", e)
print(f"  target folder exists: {TARGET.exists()}")

print("\nB) TOOLS")
tools = {t: shutil.which(t) for t in ("rclone", "aws", "s5cmd")}
for t, p in tools.items():
    print(f"  {t}: {'PRESENT '+p if p else 'missing'}")
libs = {}
for lib in ("boto3", "s3fs", "botocore"):
    try:
        m = __import__(lib); libs[lib] = getattr(m, "__version__", "?")
    except Exception:
        libs[lib] = None
for l, v in libs.items():
    print(f"  {l}: {'PRESENT '+str(v) if v else 'missing'}")

print("\nC) S3 CREDENTIAL SOURCES ON VM (presence only)")
cred_env = [k for k in os.environ if any(s in k.upper() for s in ("AWS_", "S3_", "OVH_", "OS_"))
            and any(s in k.upper() for s in ("KEY", "SECRET", "TOKEN", "ACCESS", "ENDPOINT", "PROFILE"))]
print("  relevant env vars:", cred_env or "(none)")
for f in ("~/.aws/credentials", "~/.aws/config", "~/.config/rclone/rclone.conf", "~/.s3cfg"):
    fp = Path(f).expanduser()
    print(f"  {f}: {'PRESENT' if fp.exists() else 'missing'}")

print("\nD) LIVE LISTING ATTEMPT (VM default credential chain + endpoint)")
listed = False
if libs.get("boto3"):
    import boto3
    from botocore.config import Config
    from botocore import UNSIGNED
    for mode in ("default-creds", "anonymous"):
        try:
            if mode == "anonymous":
                s3 = boto3.client("s3", endpoint_url=ENDPOINT, region_name=REGION,
                                  config=Config(signature_version=UNSIGNED, connect_timeout=15, retries={"max_attempts": 1}))
            else:
                s3 = boto3.client("s3", endpoint_url=ENDPOINT, region_name=REGION,
                                  config=Config(connect_timeout=15, retries={"max_attempts": 1}))
            r = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, Delimiter="/", MaxKeys=30)
            pfx = [p["Prefix"] for p in r.get("CommonPrefixes", [])]
            print(f"  [{mode}] SUCCESS ✓ — folders under {BUCKET}/{PREFIX}:")
            for p in pfx:
                print("     ", p)
            listed = True
            break
        except Exception as e:
            msg = str(e); code = ""
            if hasattr(e, "response"):
                code = e.response.get("Error", {}).get("Code", "")
            print(f"  [{mode}] failed: {code or type(e).__name__} — {msg[:120]}")
else:
    print("  boto3 missing — try rclone (if rclone remote configured on VM).")

print("\n" + "=" * 78)
if listed:
    print("RESULT: VM can access dpr-l0-input with ITS OWN identity → ready to download.")
else:
    print("RESULT: VM own access NONE/unverified. We need to provide keys")
    print("(copy secrets.json to VM OR set env vars) — then download cell.")

## 13) Access attempt with VM .eopf/secrets.json

Uses credentials from `the VM .eopf/secrets.json (path via $EOPF_SECRETS)` for
**live listing** on OVH buckets (`dpr-l0-input`, `dpr-s2c-commissioning-raw`, `dpr-s2-output`).
Finds (endpoint + key + secret) triples regardless of JSON schema;
does **not print values** (only which key opened which bucket).

In [ ]:
# 13) S3 access attempt with VM .eopf/secrets.json (path-style, values not printed)
import json
from pathlib import Path

SECRETS = Path(__import__("os").environ.get("EOPF_SECRETS", ""))  # export EOPF_SECRETS=/path/to/.eopf/secrets.json
BUCKETS = {
    "dpr-l0-input": "L0/Validation/S2/",
    "dpr-s2c-commissioning-raw": "",
    "dpr-s2-output": "L0/Validation/S2/",
}
DEFAULT_EP = "https://s3.sbg.io.cloud.ovh.net"

if not SECRETS.exists():
    print("!! secrets missing:", SECRETS)
    raise SystemExit

raw = json.loads(SECRETS.read_text())
print("schema type:", type(raw).__name__)

AK_NAMES = ("access_key", "access_key_id", "aws_access_key_id", "key", "key_id")
SK_NAMES = ("secret", "secret_access_key", "aws_secret_access_key", "secret_key")
EP_NAMES = ("endpoint_url", "endpoint", "host", "s3_endpoint")
RG_NAMES = ("region_name", "region")

def creds_from(dct):
    ak = next((dct[k] for k in AK_NAMES if k in dct), None)
    sk = next((dct[k] for k in SK_NAMES if k in dct), None)
    ep = next((dct[k] for k in EP_NAMES if k in dct), None)
    rg = next((dct[k] for k in RG_NAMES if k in dct), "sbg")
    return ak, sk, ep, rg

found = []
def walk(obj, label):
    if isinstance(obj, dict):
        ak, sk, ep, rg = creds_from(obj)
        if ak and sk:
            # CRITICAL: strip trailing slash
            ep = (ep or DEFAULT_EP).rstrip("/")
            found.append((label, ak, sk, ep, rg))
        for k, v in obj.items():
            walk(v, f"{label}.{k}" if label else k)
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            walk(v, f"{label}[{i}]")

walk(raw, "")
print(f"credential sets found: {len(found)}")
for label, ak, sk, ep, rg in found:
    print(f"  - [{label or 'root'}] endpoint={ep} region={rg} (key/secret hidden)")

try:
    import boto3
    from botocore.config import Config
except Exception:
    print("boto3 missing — installing")
    import subprocess, sys as _s
    subprocess.run([_s.executable, "-m", "pip", "install", "-q", "boto3"])
    import boto3
    from botocore.config import Config

def client(ak, sk, ep, rg, style):
    # CRITICAL: OVH wants path-style (NOT virtual-host)
    return boto3.client("s3", endpoint_url=ep, region_name=rg,
                        aws_access_key_id=ak, aws_secret_access_key=sk,
                        config=Config(s3={"addressing_style": style},
                                      connect_timeout=15, retries={"max_attempts": 1}))

def try_list(ak, sk, ep, rg, bucket, prefix, style):
    s3 = client(ak, sk, ep, rg, style)
    r = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, Delimiter="/", MaxKeys=40)
    return [p["Prefix"] for p in r.get("CommonPrefixes", [])]

print("\n" + "=" * 78, "\nLIVE LISTING (path-style):")
any_ok = False
for label, ak, sk, ep, rg in (found or []):
    print(f"\n### credential: [{label or 'root'}] ###")
    for bucket, prefix in BUCKETS.items():
        ok = False
        for style in ("path", "virtual"):   # path first, then virtual if needed
            try:
                pfx = try_list(ak, sk, ep, rg, bucket, prefix, style)
                names = [p.rstrip("/").split("/")[-1] for p in pfx][:12]
                print(f"  ✓ {bucket}/{prefix}  [{style}] → {len(pfx)} folders: {names}")
                any_ok = ok = True
                break
            except Exception as e:
                code = getattr(e, "response", {}).get("Error", {}).get("Code", type(e).__name__) if hasattr(e, "response") else type(e).__name__
                last = f"{code} [{style}]"
        if not ok:
            print(f"  ✗ {bucket}: {last}")

if not found:
    print("  (no key/secret found in file)")
    if isinstance(raw, dict):
        print("  top-level keys:", list(raw))

print("\n" + "=" * 78)
print("RESULT:", "At least one bucket OPENED → ready to download." if any_ok
      else "No bucket opened with these secrets.")

## 14) Each credential's own bucket — correct mapping + listing

In cell 13 keys got AccessDenied on `dpr-l0-input` because each credential is scoped to **its own** bucket.
This cell reads each entry's **bucket field** in secrets.json (name/path — not secret) and tries the key
against **its own bucket**; if no bucket field, tries root listing + a few candidates.

In [ ]:
# 14) credential -> own bucket mapping (path-style)
import json
from pathlib import Path

import boto3
from botocore.config import Config

SECRETS = Path(__import__("os").environ.get("EOPF_SECRETS", ""))  # export EOPF_SECRETS=/path/to/.eopf/secrets.json
raw = json.loads(SECRETS.read_text())

AK = ("access_key", "access_key_id", "aws_access_key_id", "key", "key_id")
SK = ("secret", "secret_access_key", "aws_secret_access_key", "secret_key")
EP = ("endpoint_url", "endpoint", "host", "s3_endpoint")
RG = ("region_name", "region")
BK = ("bucket", "bucket_name", "bucketname", "container")  # bucket field name candidates

def val(d, names):
    return next((d[k] for k in names if k in d), None)

# each entry: label -> (ak,sk,ep,rg, bucket, all-field-names)
entries = []
for label, e in (raw.items() if isinstance(raw, dict) else []):
    if not isinstance(e, dict):
        continue
    ak, sk = val(e, AK), val(e, SK)
    if not (ak and sk):
        continue
    ep = (val(e, EP) or "https://s3.sbg.io.cloud.ovh.net").rstrip("/")
    rg = val(e, RG) or "sbg"
    bucket = val(e, BK)
    # bucket may also be in path/prefix field
    other = {k: v for k, v in e.items() if k not in AK + SK and not any(s in k.lower() for s in ("secret", "key", "token"))}
    entries.append((label, ak, sk, ep, rg, bucket, other))

print("secrets.json entries (non-secret fields):")
for label, ak, sk, ep, rg, bucket, other in entries:
    print(f"  [{label}] bucket={bucket!r}  other_fields={other}")

def lister(ak, sk, ep, rg):
    return boto3.client("s3", endpoint_url=ep, region_name=rg,
                        aws_access_key_id=ak, aws_secret_access_key=sk,
                        config=Config(s3={"addressing_style": "path"},
                                      connect_timeout=15, retries={"max_attempts": 1}))

print("\n" + "=" * 78, "\nEACH CREDENTIAL IN ITS OWN BUCKET:")
for label, ak, sk, ep, rg, bucket, other in entries:
    print(f"\n### [{label}] ###")
    s3 = lister(ak, sk, ep, rg)
    # 1) list_buckets — which buckets does this credential see?
    try:
        b = s3.list_buckets()
        names = [x["Name"] for x in b.get("Buckets", [])]
        print(f"  list_buckets → {names}")
    except Exception as e:
        print(f"  list_buckets ✗ ({getattr(e,'response',{}).get('Error',{}).get('Code', type(e).__name__)})")
        names = []
    # 2) list declared bucket or visible buckets
    candidates = [c for c in ([bucket] + names) if c]
    seen = set()
    for cand in candidates:
        if cand in seen:
            continue
        seen.add(cand)
        try:
            r = s3.list_objects_v2(Bucket=cand, Delimiter="/", MaxKeys=40)
            pfx = [p["Prefix"].rstrip("/").split("/")[-1] for p in r.get("CommonPrefixes", [])]
            files = [o["Key"] for o in r.get("Contents", [])][:5]
            print(f"  ✓ {cand} → folders={pfx[:15]} files={files}")
        except Exception as e:
            code = getattr(e, "response", {}).get("Error", {}).get("Code", type(e).__name__)
            print(f"  ✗ {cand}: {code}")

print("\n" + "=" * 78)
print("Above, ✓ (credential, bucket) pairs = data we can download.")

## 15) dpr-common discovery — S2 MSI dataset + ADF (calibration) epochs

The `s3input` key opens `dpr-common`. Contains `ADF-S02MSI`, `ADFstatic`, `ADFdynamic` (calibration)
and `S2AMSIdataset/S2BMSIdataset/S2CMSIdataset`. This cell: (A) extracts **R2EQOG/R2ABCA epochs**
(with production dates) from ADF folders — is the temporal-valid GIPP we need here? (B) lists each
satellite's MSI dataset layout + sample product names (acquisition dates).

In [ ]:
# 15) dpr-common: S2 MSI dataset + ADF epoch scan (s3input key)
import json
import re
from pathlib import Path

import boto3
from botocore.config import Config

SECRETS = Path(__import__("os").environ.get("EOPF_SECRETS", ""))  # export EOPF_SECRETS=/path/to/.eopf/secrets.json
raw = json.loads(SECRETS.read_text())
e = raw["s3input"]
ak = e.get("access_key") or e.get("access_key_id") or e.get("key")
sk = e.get("secret") or e.get("secret_access_key")
ep = (e.get("endpoint_url") or "https://s3.sbg.io.cloud.ovh.net").rstrip("/")
rg = e.get("region_name", "sbg")
s3 = boto3.client("s3", endpoint_url=ep, region_name=rg,
                  aws_access_key_id=ak, aws_secret_access_key=sk,
                  config=Config(s3={"addressing_style": "path"}, connect_timeout=15,
                                retries={"max_attempts": 2}))
BUCKET = "dpr-common"

def ls(prefix, delim="/", maxk=200):
    pfx, keys = [], []
    tok = None
    while True:
        kw = dict(Bucket=BUCKET, Prefix=prefix, Delimiter=delim, MaxKeys=1000)
        if tok:
            kw["ContinuationToken"] = tok
        r = s3.list_objects_v2(**kw)
        pfx += [p["Prefix"] for p in r.get("CommonPrefixes", [])]
        keys += [(o["Key"], o["Size"]) for o in r.get("Contents", [])]
        if r.get("IsTruncated") and len(keys) < maxk:
            tok = r.get("NextContinuationToken")
        else:
            break
    return pfx, keys

def _d(t):
    return f"{t[:4]}-{t[4:6]}-{t[6:8]}" if t and len(t) >= 8 else t

# --- A) ADF folders: R2EQOG/R2ABCA epochs ---
GIP = re.compile(r"(R2EQOG|R2ABCA|RABCA|EQOG)[^/]*?(\d{8})T?\d*", re.I)
ADF_GEN = re.compile(r"(20\d{6})")
for adf in ("ADF-S02MSI/", "ADFstatic/", "ADFdynamic/"):
    print("=" * 84)
    print("dpr-common/" + adf)
    try:
        sub, keys = ls(adf)
        if sub:
            print("  subfolders:", [p.rstrip("/").split("/")[-1] for p in sub][:30])
        # fetch files too (not recursive, one level deeper)
        allkeys = list(keys)
        for s in sub[:40]:
            _, k2 = ls(s, maxk=400)
            allkeys += k2
        # capture R2EQOG/R2ABCA
        eqog = {}
        for k, sz in allkeys:
            base = k.split("/")[-1]
            if re.search(r"R2EQOG|EQOG", base, re.I) or re.search(r"R2ABCA|RABCA", base, re.I):
                typ = "R2EQOG" if re.search(r"EQOG", base, re.I) else "R2ABCA"
                dts = ADF_GEN.findall(base)
                eqog.setdefault((typ, tuple(dts)), (base, sz))
        if eqog:
            print("  >>> RADIOMETRIC ADF found:")
            for (typ, dts), (base, sz) in sorted(eqog.items()):
                print(f"     {typ}  dates={[_d(d) for d in dts]}  ({sz/1024:.0f} KB)  {base[:70]}")
        else:
            print(f"  (no R2EQOG/R2ABCA name; total {len(allkeys)} files)")
            for k, sz in allkeys[:8]:
                print("      sample:", k.split("/")[-1])
    except Exception as ex:
        print("  ✗", getattr(ex, "response", {}).get("Error", {}).get("Code", type(ex).__name__))

# --- B) MSI dataset layout + sample products (acquisition dates) ---
for ds in ("S2AMSIdataset/", "S2BMSIdataset/", "S2CMSIdataset/"):
    print("=" * 84)
    print("dpr-common/" + ds)
    try:
        sub, keys = ls(ds)
        print("  subfolders:", [p.rstrip("/").split("/")[-1] for p in sub][:30] or "(none)")
        # acquisition date in product names
        prods = [p.rstrip("/").split("/")[-1] for p in sub] + [k.split("/")[-1] for k, _ in keys]
        dated = sorted({m for pr in prods for m in re.findall(r"20\d{2}[01]\d[0-3]\dT?\d*", pr)})[:15]
        if dated:
            print("  acquisition/date stamps in products:", dated)
        for k, sz in keys[:5]:
            print(f"    dosya: {k.split('/')[-1]}  ({sz/2**20:.1f} MB)")
    except Exception as ex:
        print("  ✗", getattr(ex, "response", {}).get("Error", {}).get("Code", type(ex).__name__))

print("\n" + "=" * 84)
print("R2EQOG epoch(s) in ADF folders + MSI dataset acquisition dates are above.")

## 16) S2A full-chain hunt — L0+L1A+L1B+L1C for the same acquisition?

To validate the pipeline we need a full level chain (L0 RAW → L1A → L1B →
L1C) for the **same acquisition** on S2A + applied ADFs. This cell recursively scans
`dpr-common/S2AMSIdataset/` (scene folders turkey/quebec/lake + tar bundles); groups products by
**datatake/sensing date** and shows which levels coexist on which dates + sizes.

In [ ]:
# 16) dpr-common/S2AMSIdataset: same-acquisition full-chain scan
import json
import re
from collections import defaultdict
from pathlib import Path

import boto3
from botocore.config import Config

SECRETS = Path(__import__("os").environ.get("EOPF_SECRETS", ""))  # export EOPF_SECRETS=/path/to/.eopf/secrets.json
e = json.loads(SECRETS.read_text())["s3input"]
ak = e.get("access_key") or e.get("access_key_id") or e.get("key")
sk = e.get("secret") or e.get("secret_access_key")
ep = (e.get("endpoint_url") or "https://s3.sbg.io.cloud.ovh.net").rstrip("/")
s3 = boto3.client("s3", endpoint_url=ep, region_name=e.get("region_name", "sbg"),
                  aws_access_key_id=ak, aws_secret_access_key=sk,
                  config=Config(s3={"addressing_style": "path"}, connect_timeout=15,
                                retries={"max_attempts": 2}))
BUCKET = "dpr-common"
ROOT = "S2AMSIdataset/"

def ls_all(prefix, cap=8000):
    keys = []
    tok = None
    while True:
        kw = dict(Bucket=BUCKET, Prefix=prefix, MaxKeys=1000)
        if tok:
            kw["ContinuationToken"] = tok
        r = s3.list_objects_v2(**kw)
        keys += [(o["Key"], o["Size"]) for o in r.get("Contents", [])]
        if r.get("IsTruncated") and len(keys) < cap:
            tok = r.get("NextContinuationToken")
        else:
            break
    return keys

def _d(t):
    return f"{t[:4]}-{t[4:6]}-{t[6:8]}" if t and len(t) >= 8 else t

def level_of(name):
    for L in ("L1C", "L1B", "L1A", "L2A", "L0P", "L0__", "L0_", "L0"):
        if L in name:
            return "L0" if L.startswith("L0") else L
    return None

def sensing_of(name):
    # MSILxx_<sensing>  or  _V<sensing>T  or  _S<sensing>T
    for pat in (r"MSIL[0-9][A-C_]{0,2}_(\d{8})T", r"_V(\d{8})T", r"_S(\d{8})T", r"DS_\w+_(\d{8})T"):
        m = re.search(pat, name)
        if m:
            return m.group(1)
    return None

# 1) top-level folder/bundle inventory
top = s3.list_objects_v2(Bucket=BUCKET, Prefix=ROOT, Delimiter="/")
folders = [p["Prefix"] for p in top.get("CommonPrefixes", [])]
toptars = [(o["Key"], o["Size"]) for o in top.get("Contents", [])]
print("top folders:", [f.rstrip("/").split("/")[-1] for f in folders])
print("top tar/files:")
for k, sz in sorted(toptars, key=lambda x: -x[1])[:15]:
    print(f"   {k.split('/')[-1]:45s} {sz/2**30:6.1f} GB")

# 2) scan each scene folder, group by level+date
def analyze(prefix, label):
    keys = ls_all(prefix)
    # capture product roots (.SAFE / .zarr / DS / .tar)
    prods = defaultdict(lambda: [None, 0])  # (level,sensing) -> [sample_name, total_size]
    for k, sz in keys:
        rel = k[len(prefix):]
        root = rel.split("/")[0] if "/" in rel else rel
        lvl = level_of(root) or level_of(k)
        sen = sensing_of(root) or sensing_of(k)
        key = (lvl, sen)
        cur = prods[key]
        if cur[0] is None:
            cur[0] = root
        cur[1] += sz
    print(f"\n{'='*84}\n[{label}]  ({len(keys)} objects)")
    # group by date: which levels share the same sensing?
    bydate = defaultdict(dict)
    for (lvl, sen), (ex, sz) in prods.items():
        if sen:
            bydate[sen][lvl or "?"] = (ex, sz)
    for sen in sorted(bydate):
        levels = bydate[sen]
        lv = sorted(levels)
        chain = "FULL CHAIN ✓" if {"L0", "L1A", "L1B"} <= set(levels) else \
                ("partial" if len(levels) > 1 else "single")
        print(f"  acquisition {_d(sen)}: levels={lv}  [{chain}]")
        for l in lv:
            ex, sz = levels[l]
            print(f"        {l:5s} {sz/2**20:8.1f} MB  {ex[:60]}")
    # tarihsiz ama seviyeli
    notime = {l for (l, s) in prods if s is None and l}
    if notime:
        print("  (tarihsiz seviye izleri:", notime, ")")

for f in folders:
    analyze(f, f.rstrip("/").split("/")[-1])
# group tar bundles by date too (53785 etc datatake)
print("\n" + "=" * 84)
print("TAR BUNDLES (by datatake number — same number = same acquisition chain):")
dt = defaultdict(list)
for k, sz in toptars:
    n = k.split("/")[-1]
    m = re.search(r"(\d{4,6})[-_]", n)
    dt[m.group(1) if m else n].append((n, sz))
for num, items in sorted(dt.items()):
    print(f"  datatake {num}:")
    for n, sz in items:
        print(f"      {n:45s} {sz/2**30:6.2f} GB  {'<L0>' if 'L0' in n else ''}")

print("\n" + "=" * 84)
print("If an acquisition has 'FULL CHAIN ✓' → that is ideal S2A data for pipeline validation.")

## 17) quebec + turkey full content + applied GIPP_List

Lists **each level's** (L0/L1A/L1B/L1C/L2A) product name + size, confirms **L1A exists**
(generator input), and reads **applied R2EQOG/
R2ABCA epoch** (GIPP_List) from each scene's datastrip `MTD_DS.xml` → which calibration ESA actually applied.
Finally: download manifest + total size.

In [ ]:
# 17) quebec/turkey manifest + applied GIPP
import json
import re
from collections import defaultdict
from pathlib import Path

import boto3
from botocore.config import Config

SECRETS = Path(__import__("os").environ.get("EOPF_SECRETS", ""))  # export EOPF_SECRETS=/path/to/.eopf/secrets.json
e = json.loads(SECRETS.read_text())["s3input"]
s3 = boto3.client("s3", endpoint_url=(e.get("endpoint_url") or "https://s3.sbg.io.cloud.ovh.net").rstrip("/"),
                  region_name=e.get("region_name", "sbg"),
                  aws_access_key_id=e.get("access_key") or e.get("access_key_id") or e.get("key"),
                  aws_secret_access_key=e.get("secret") or e.get("secret_access_key"),
                  config=Config(s3={"addressing_style": "path"}, connect_timeout=15, retries={"max_attempts": 2}))
BUCKET = "dpr-common"

def ls_all(prefix, cap=20000):
    keys, tok = [], None
    while True:
        kw = dict(Bucket=BUCKET, Prefix=prefix, MaxKeys=1000)
        if tok: kw["ContinuationToken"] = tok
        r = s3.list_objects_v2(**kw)
        keys += [(o["Key"], o["Size"]) for o in r.get("Contents", [])]
        if r.get("IsTruncated") and len(keys) < cap: tok = r.get("NextContinuationToken")
        else: break
    return keys

def level_of(n):
    for L, tag in (("MSIL1A", "L1A"), ("MSIL1B", "L1B"), ("MSIL1C", "L1C"), ("MSIL2A", "L2A"),
                   ("MSI_L0", "L0"), ("MSIL0", "L0"), ("_L1A_", "L1A"), ("_L1B_", "L1B"),
                   ("_L0__", "L0"), ("L1B_DS", "L1B"), ("L1A_DS", "L1A")):
        if L in n: return tag
    return None

GIP = re.compile(r"S2[AB]_OPER_GIP_(?P<t>\w{6})_\w+?_{1,2}(?P<c>\d{8})T\d{6}(?:_V(?P<vs>\d{8}))?")

def _d(t): return f"{t[:4]}-{t[4:6]}-{t[6:8]}" if t and len(t) >= 8 else t

for scene in ("quebec/", "turkey/"):
    print("=" * 88, f"\nSCENE: {scene}")
    keys = ls_all(f"S2AMSIdataset/{scene}")
    # product root = first segment under scene
    prod = defaultdict(lambda: [None, 0])
    for k, sz in keys:
        rel = k[len(f"S2AMSIdataset/{scene}"):]
        root = rel.split("/")[0]
        p = prod[root]; p[1] += sz
        if p[0] is None: p[0] = k
    total = 0
    for root in sorted(prod):
        ex, sz = prod[root]
        total += sz
        lvl = level_of(root) or level_of(ex) or "?"
        print(f"  [{lvl:4s}] {sz/2**20:8.1f} MB  {root[:74]}")
    print(f"  --- TOTAL: {total/2**30:.2f} GB ---")

    # applied GIPP: find datastrip MTD_DS.xml, download, extract R2EQOG/R2ABCA
    dsxml = [k for k, _ in keys if k.endswith("MTD_DS.xml") or (("DATASTRIP" in k or "_DS_" in k) and k.endswith(".xml"))]
    print(f"  datastrip xml candidate count: {len(dsxml)}")
    got = False
    for x in dsxml[:4]:
        try:
            body = s3.get_object(Bucket=BUCKET, Key=x)["Body"].read().decode("utf-8", "replace")
        except Exception as ex:
            continue
        gipps = {}
        for m in GIP.finditer(body):
            gipps.setdefault(m["t"], (m["c"], m["vs"]))
        if gipps:
            r2 = gipps.get("R2EQOG"); ab = gipps.get("R2ABCA")
            print(f"  >>> UYGULANAN GIPP ({x.split('/')[-1]}):")
            if r2: print(f"        R2EQOG production={_d(r2[0])} valid={_d(r2[1])}")
            if ab: print(f"        R2ABCA production={_d(ab[0])} valid={_d(ab[1])}")
            print(f"        (toplam {len(gipps)} GIPP tipi datastrip'te)")
            got = True
            break
    if not got:
        print("  (could not read GIPP_List in datastrip — L0/L1 xml may differ)")

print("\n" + "=" * 88)
print("If L1A + L1B present, generator input is ready. R2EQOG production date = calibration")
print("actually applied to that scene → use exactly this in reverse chain (full temporal match).")

## 18) DOWNLOAD — turkey full chain → data-store

Downloads required files from `dpr-common/S2AMSIdataset/turkey/` (**L1B + L0 + L1C + L2A + CAMS**, excludes large ECMWF
meteo ADFs) to `/home/jovyan/validation-data/data-store/dpr-common-validation/turkey/` on the VM.
`SCENE` and `SKIP` change scene/filter. With `DRY_RUN=True`
only plan+size is shown first; set `DRY_RUN=False` and re-run to download.

In [ ]:
# 18) download turkey full chain
import json
import os
from pathlib import Path

import boto3
from botocore.config import Config
from boto3.s3.transfer import TransferConfig

# ---- SETTINGS ----
SCENE = "turkey"                       # can use "quebec" (but L1B is 42GB)
SKIP_SUBSTR = ("ADF_ECMW",)            # skip dev meteo ADFs; use () to download everything
DEST = Path(f"/home/jovyan/validation-data/data-store/dpr-common-validation/{SCENE}")
DRY_RUN = False                         # plan first; set False to download
# -----------------

SECRETS = Path(__import__("os").environ.get("EOPF_SECRETS", ""))  # export EOPF_SECRETS=/path/to/.eopf/secrets.json
e = json.loads(SECRETS.read_text())["s3input"]
s3 = boto3.client("s3", endpoint_url=(e.get("endpoint_url") or "https://s3.sbg.io.cloud.ovh.net").rstrip("/"),
                  region_name=e.get("region_name", "sbg"),
                  aws_access_key_id=e.get("access_key") or e.get("access_key_id") or e.get("key"),
                  aws_secret_access_key=e.get("secret") or e.get("secret_access_key"),
                  config=Config(s3={"addressing_style": "path"}, connect_timeout=20,
                                retries={"max_attempts": 3}))
BUCKET = "dpr-common"
PREFIX = f"S2AMSIdataset/{SCENE}/"
xfer = TransferConfig(multipart_threshold=64 * 2**20, multipart_chunksize=64 * 2**20,
                      max_concurrency=8, use_threads=True)

# 1) object list
keys, tok = [], None
while True:
    kw = dict(Bucket=BUCKET, Prefix=PREFIX, MaxKeys=1000)
    if tok: kw["ContinuationToken"] = tok
    r = s3.list_objects_v2(**kw)
    keys += [(o["Key"], o["Size"]) for o in r.get("Contents", [])]
    if r.get("IsTruncated"): tok = r.get("NextContinuationToken")
    else: break

# 2) filter + plan
plan = [(k, sz) for k, sz in keys if not any(s in k.split("/")[-1] for s in SKIP_SUBSTR)]
skipped = [(k, sz) for k, sz in keys if any(s in k.split("/")[-1] for s in SKIP_SUBSTR)]
tot = sum(sz for _, sz in plan)
print(f"SCENE: {SCENE}   target: {DEST}")
print(f"to download: {len(plan)} files, {tot/2**30:.2f} GB")
for k, sz in sorted(plan, key=lambda x: -x[1]):
    print(f"   {sz/2**20:9.1f} MB  {k.split('/')[-1][:70]}")
if skipped:
    print(f"skipped ({sum(s for _,s in skipped)/2**30:.1f} GB):", [k.split('/')[-1][:40] for k,_ in skipped])

# 3) disk check
import shutil
probe = DEST
while not probe.exists():            # check nearest EXISTING parent of DEST
    probe = probe.parent
free = shutil.disk_usage(probe).free
print(f"\nfree disk ({probe}): {free/2**30:.1f} GB  →  {'OK' if free > tot*1.2 else 'RISKY!'}")

if DRY_RUN:
    print("\n[DRY_RUN] no download. Set DRY_RUN=False in cell and re-run to download.")
else:
    DEST.mkdir(parents=True, exist_ok=True)
    done = 0
    for i, (k, sz) in enumerate(sorted(plan, key=lambda x: x[1]), 1):
        rel = k[len(PREFIX):]
        out = DEST / rel
        out.parent.mkdir(parents=True, exist_ok=True)
        if out.exists() and out.stat().st_size == sz:
            print(f"  [{i}/{len(plan)}] already present, skip: {rel[:60]}")
            done += sz; continue
        print(f"  [{i}/{len(plan)}] {sz/2**20:.0f} MB ← {rel[:60]} ...", flush=True)
        s3.download_file(BUCKET, k, str(out), Config=xfer)
        done += sz
        print(f"      ✓  ({done/2**30:.2f}/{tot/2**30:.2f} GB)")
    print(f"\nDONE — {DEST} ({done/2**30:.2f} GB downloaded)")
    print("Next: open L1B.tar / L0 tar → read applied R2EQOG epoch from datastrip GIPP_List.")

## 19) DECISION — do we have matching GIPP for the downloaded turkey product?

Extracts datastrip metadata from downloaded turkey L1B/L0 tars (without extracting 12 GB), reads **ESA's
actually applied R2EQOG/R2ABCA epoch** and compares with **GIPP epochs we have**
→ decision on "do we have matching GIPP, or which imagery can we process".

In [ ]:
# 19) applied GIPP from turkey datastrip + compare with what we have
import re
import tarfile
from pathlib import Path

DEST = Path("/home/jovyan/validation-data/data-store/dpr-common-validation/turkey")

# AVAILABLE accessible R2EQOG epochs (files present/downloadable)
AVAILABLE = {
    "2020-03-10": "pipeline data-store (bundled, XML, 13 band)",
    "2022-09-08": "dpr-common/S2AMSIdataset loose GIPP (TGZ, 13 band)",
    "2024-04-17": "dpr-common/ADF-S02MSI (EOPF json)",
}

GIP = re.compile(r"S2[AB]_OPER_GIP_(?P<t>\w{6})_\w+?_{1,2}(?P<c>\d{8})T\d{6}(?:_V(?P<vs>\d{8}))?")
def _d(t): return f"{t[:4]}-{t[4:6]}-{t[6:8]}" if t and len(t) >= 8 else t

tars = sorted(DEST.rglob("*.tar")) if DEST.exists() else []
print("target:", DEST, "| tars found:", [t.name for t in tars] or "(NONE — download finished?)")

applied = {}  # type -> set(production dates)
for tf in tars:
    try:
        with tarfile.open(tf) as tar:
            members = tar.getmembers()
    except Exception as ex:
        print(f"  {tf.name}: could not open ({ex}) — download may be incomplete")
        continue
    # find datastrip metadata members
    ds = [m for m in members if m.isfile() and (
          m.name.endswith("MTD_DS.xml") or re.search(r"(DATASTRIP|_DS_).*\.xml$", m.name))]
    print(f"  {tf.name}: {len(members)} members, {len(ds)} datastrip-xml candidates")
    for m in ds[:6]:
        try:
            body = tar.extractfile(m).read().decode("utf-8", "replace") if False else None
        except Exception:
            body = None
        # reopen tar if closed
        try:
            with tarfile.open(tf) as tar2:
                body = tar2.extractfile(m.name).read().decode("utf-8", "replace")
        except Exception as ex:
            print(f"     {m.name.split('/')[-1]}: unreadable ({ex})"); continue
        hits = {}
        for g in GIP.finditer(body):
            hits.setdefault(g["t"], (g["c"], g["vs"]))
        if hits:
            print(f"     >>> {m.name.split('/')[-1]}: {len(hits)} GIPP")
            for typ in ("R2EQOG", "R2ABCA"):
                if typ in hits:
                    c, vs = hits[typ]
                    applied.setdefault(typ, set()).add(c)
                    print(f"         {typ} production={_d(c)} valid={_d(vs)}")

print("\n" + "=" * 80, "\nDECISION:")
if not applied:
    print("  could not read GIPP from turkey datastrip (download incomplete or different format).")
    print("  → re-run this cell when download completes.")
else:
    for typ, dates in applied.items():
        for c in sorted(dates):
            cd = _d(c)
            exact = cd in AVAILABLE
            # nearest available
            def gap(a):
                return abs(int(a.replace("-", "")) - int(c))
            nearest = min(AVAILABLE, key=lambda a: abs(int(a.replace("-","")) - int(cd.replace("-",""))))
            gapdays = abs(int(nearest.replace("-","")) - int(cd.replace("-","")))
            print(f"  {typ} applied={cd}:")
            if exact:
                print(f"     ✓✓ EXACT MATCHING GIPP AVAILABLE → {AVAILABLE[cd]}")
            else:
                print(f"     ✗ no exact match. Nearest available: {nearest} ({AVAILABLE[nearest]})")
                print(f"        (~{gapdays//10000} year gap — weak temporal match)")
    print("\n  Summary: if 'EXACT MATCH' we can process this turkey product ESA-only + temporal-consistent.")
    print("  Else: either download matching epoch separately, or pick another L1B product")
    print("  acquired near our epoch (e.g. autumn 2022 acquisition for 2022-09 GIPP).")

## 20) turkey L1B.tar — structure + applied R2EQOG (actual reverse-chain GIPP)

L0 datastrip gave R2ABCA=2018-08-07 but not R2EQOG (equalization is an L1 step). Reverse chain's
R2EQOG lives in **L1B datastrip**. This cell shows L1B.tar member names + searches **all xml
members** for GIPP/R2EQOG → real R2EQOG epoch applied to turkey L1B.

In [ ]:
# 20) extract R2EQOG/GIPP from L1B.tar xml members
import re
import tarfile
from pathlib import Path

DEST = Path("/home/jovyan/validation-data/data-store/dpr-common-validation/turkey")
L1B = next(iter(sorted(DEST.rglob("*_L1B.tar"))), None)
print("L1B tar:", L1B)
if not L1B:
    print("L1B.tar missing"); raise SystemExit

GIP = re.compile(r"S2[AB]_OPER_GIP_(?P<t>\w{6})_\w+?_{1,2}(?P<c>\d{8})T\d{6}(?:_V(?P<vs>\d{8}))?")
def _d(t): return f"{t[:4]}-{t[4:6]}-{t[6:8]}" if t and len(t) >= 8 else t

with tarfile.open(L1B) as tar:
    members = tar.getmembers()
    names = [m.name for m in members if m.isfile()]
    xmls = [n for n in names if n.lower().endswith(".xml")]
    print(f"{len(names)} file members, {len(xmls)} xml")
    print("sample member names (first 20):")
    for n in names[:20]:
        print("   ", n)
    print("xml members:")
    for n in xmls[:30]:
        print("   ", n)

    # search all xml for GIPP
    applied = {}
    for n in xmls:
        try:
            body = tar.extractfile(n).read().decode("utf-8", "replace")
        except Exception:
            continue
        for g in GIP.finditer(body):
            applied.setdefault(g["t"], set()).add((g["c"], g["vs"]))

print("\n" + "=" * 80)
if applied:
    print("GIPP types found in L1B:")
    for t in sorted(applied):
        for c, vs in sorted(applied[t]):
            mark = "  <<< NUC/reverse-chain" if t == "R2EQOG" else ""
            print(f"   {t}  production={_d(c)} valid={_d(vs)}{mark}")
    r2 = applied.get("R2EQOG")
    if r2:
        c = sorted(r2)[0][0]
        AVAIL = {"2020-03-10", "2022-09-08", "2024-04-17"}
        print(f"\n>>> R2EQOG applied to turkey L1B = {_d(c)}")
        print("    available:", "YES ✓✓" if _d(c) in AVAIL else "NO ✗ (nearest: 2020-03-10)")
else:
    print("No GIPP reference in L1B xml — L1B may be different/intermediate format.")
    print("(In this case R2EQOG epoch is not embedded; we supply GIPP externally for reverse chain.)")

## 21) L1B granule (nested tar) → definitive applied R2EQOG epoch

Granules inside L1B.tar are their own tars. Opens one granule in memory and reads metadata/xml for
**R2EQOG epoch applied to turkey L1B (N05.11, 2025 reprocess)** and checks match with our
EOPF R2EQOG 2024-04-17 — definitive decision.

In [ ]:
# 21) R2EQOG from nested granule tar
import io
import re
import tarfile
from pathlib import Path

DEST = Path("/home/jovyan/validation-data/data-store/dpr-common-validation/turkey")
L1B = next(iter(sorted(DEST.rglob("*_L1B.tar"))), None)
GIP = re.compile(r"S2[AB]_OPER_GIP_(?P<t>\w{6})_\w+?_{1,2}(?P<c>\d{8})T\d{6}(?:_V(?P<vs>\d{8}))?")
def _d(t): return f"{t[:4]}-{t[4:6]}-{t[6:8]}" if t and len(t) >= 8 else t

applied = {}
with tarfile.open(L1B) as outer:
    gran = [m for m in outer.getmembers() if m.isfile() and m.name.endswith(".tar")]
    print(f"{len(gran)} granule tars. Opening first 3...")
    for m in gran[:3]:
        data = outer.extractfile(m).read()
        try:
            with tarfile.open(fileobj=io.BytesIO(data)) as inner:
                inames = [x.name for x in inner.getmembers() if x.isfile()]
                if m is gran[0]:
                    print(f"  granule inner members ({len(inames)}):")
                    for n in inames[:25]:
                        print("     ", n)
                for n in inames:
                    if n.lower().endswith((".xml", ".json", ".gml", ".hdr")):
                        body = inner.extractfile(n).read().decode("utf-8", "replace")
                        for g in GIP.finditer(body):
                            applied.setdefault(g["t"], set()).add((g["c"], g["vs"]))
        except Exception as ex:
            print("  granule could not open:", ex)

print("\n" + "=" * 80)
if applied:
    print("GIPP found in granule:")
    for t in sorted(applied):
        for c, vs in sorted(applied[t]):
            mk = "  <<< NUC" if t == "R2EQOG" else ""
            print(f"   {t} production={_d(c)} valid={_d(vs)}{mk}")
    r2 = applied.get("R2EQOG")
    if r2:
        c = _d(sorted(r2)[0][0])
        print(f"\n>>> R2EQOG applied to turkey L1B = {c}")
        print("    vs EOPF 2024-04-17:", "MATCHES ✓✓" if c == "2024-04-17"
              else f"no match — we do not have {c}")
else:
    print("No GIPP reference in granule metadata either.")
    print("→ R2EQOG epoch not recorded in product; we choose and supply GIPP for reverse chain")
    print("  (L1B baseline N05.11 → most consistent is EOPF R2EQOG 2024-04-17).")

## 22) Download EOPF R2EQOG 2024-04-17 ADF to VM data-store

Downloads ESA equalization/NUC ADF (EOPF `.json`, pipeline-readable) from `dpr-common/ADF-S02MSI`
to `/home/jovyan/data-store/esa-adf/`: S2A+S2B **REQOG** (62 MB×2) + **RABCA** (2024-04-17 &
2024-07-04). This is the real ESA NUC source that will replace derived `caldb` NUC. (~124 MB)

In [ ]:
# 22) EOPF R2EQOG/RABCA ADF -> VM data-store
import json
from pathlib import Path

import boto3
from botocore.config import Config
from boto3.s3.transfer import TransferConfig

SECRETS = Path(__import__("os").environ.get("EOPF_SECRETS", ""))  # export EOPF_SECRETS=/path/to/.eopf/secrets.json
e = json.loads(SECRETS.read_text())["s3input"]
s3 = boto3.client("s3", endpoint_url=(e.get("endpoint_url") or "https://s3.sbg.io.cloud.ovh.net").rstrip("/"),
                  region_name=e.get("region_name", "sbg"),
                  aws_access_key_id=e.get("access_key") or e.get("access_key_id") or e.get("key"),
                  aws_secret_access_key=e.get("secret") or e.get("secret_access_key"),
                  config=Config(s3={"addressing_style": "path"}, connect_timeout=20, retries={"max_attempts": 3}))
BUCKET, PREFIX = "dpr-common", "ADF-S02MSI/"
DEST = Path("/home/jovyan/data-store/esa-adf")
DEST.mkdir(parents=True, exist_ok=True)
xfer = TransferConfig(multipart_threshold=32 * 2**20, multipart_chunksize=32 * 2**20, max_concurrency=8)

FILES = [
    "S2A_ADF_REQOG_20240417T000000_21000101T000000_20240417T000000.json",
    "S2B_ADF_REQOG_20240417T000000_21000101T000000_20240417T000000.json",
    "S2A_ADF_RABCA_20240417T000000_21000101T000000_20240417T000000.json",
    "S2B_ADF_RABCA_20240417T000000_21000101T000000_20240417T000000.json",
    "S2A_ADF_RABCA_20240704T000000_21000101T000000_20240704T000000.json",
    "S2B_ADF_RABCA_20240704T000000_21000101T000000_20240704T000000.json",
]
for f in FILES:
    out = DEST / f
    if out.exists() and out.stat().st_size > 0:
        print("already present:", f); continue
    try:
        s3.download_file(BUCKET, PREFIX + f, str(out), Config=xfer)
        print(f"✓ {out.stat().st_size/2**20:.1f} MB  {f}")
    except Exception as ex:
        print("✗", f, "-", getattr(ex, "response", {}).get("Error", {}).get("Code", type(ex).__name__))

print("\ndownloaded ->", DEST)
for p in sorted(DEST.glob("*.json")):
    print(f"   {p.stat().st_size/2**20:7.1f} MB  {p.name}")
print("\nEOPF R2EQOG: data_vars=per-band coeff_a..d × 12 detectors. ESA NUC source for reverse chain.")